In [4]:
import requests
import datetime
import pandas as pd
import time

# --- 1. THE INGESTION ENGINE ---
def fetch_drought_features(lat, lon):
    end_date = datetime.datetime.now() - datetime.timedelta(days=2) 
    start_date = end_date - datetime.timedelta(days=7)
    
    start_str = start_date.strftime('%Y%m%d')
    end_str = end_date.strftime('%Y%m%d')
    
    parameters = "PRECTOTCORR,PS,QV2M,T2M,T2MDEW,T2MWET,T2M_MAX,T2M_MIN,T2M_RANGE,TS,WS10M"
    
    url = (f"https://power.larc.nasa.gov/api/temporal/daily/point?"
           f"parameters={parameters}&community=AG&longitude={lon}&latitude={lat}"
           f"&start={start_str}&end={end_str}&format=JSON")
    
    response = requests.get(url)
    if response.status_code != 200:
        raise Exception(f"Failed to fetch data. Status code: {response.status_code}")
        
    data = response.json()['properties']['parameter']
    features = {}
    for param in parameters.split(','):
        daily_values = list(data[param].values())
        valid_values = [v for v in daily_values if v != -999.0]
        features[param] = sum(valid_values) / len(valid_values) if valid_values else 0.0
    return features

# --- 2. THE 100-REGION PLANETARY MATRIX ---
global_training_zones = [
    # Extreme Deserts (Score 4)
    {"region": "Sahara Desert, Chad", "lat": 18.0, "lon": 19.0, "actual_drought_score": 4},
    {"region": "Atacama Desert, Chile", "lat": -23.86, "lon": -69.13, "actual_drought_score": 4},
    {"region": "Rub' al Khali, Saudi Arabia", "lat": 20.0, "lon": 50.0, "actual_drought_score": 4},
    {"region": "Namib Desert, Namibia", "lat": -24.75, "lon": 15.27, "actual_drought_score": 4},
    {"region": "Kalahari Desert, Botswana", "lat": -23.9, "lon": 22.1, "actual_drought_score": 4},
    {"region": "Simpson Desert, Australia", "lat": -24.5, "lon": 137.4, "actual_drought_score": 4},
    {"region": "Thar Desert, India", "lat": 26.9, "lon": 70.9, "actual_drought_score": 4},
    {"region": "Taklamakan Desert, China", "lat": 38.9, "lon": 82.5, "actual_drought_score": 4},
    {"region": "Mojave Desert, USA", "lat": 35.0, "lon": -115.5, "actual_drought_score": 4},
    {"region": "Karakum Desert, Turkmenistan", "lat": 39.0, "lon": 60.0, "actual_drought_score": 4},
    
    # Active/Historic Severe Drought Zones (Score 3-4)
    {"region": "Horn of Africa, Somalia", "lat": 5.15, "lon": 46.19, "actual_drought_score": 4},
    {"region": "California Central Valley, USA", "lat": 36.77, "lon": -119.41, "actual_drought_score": 3},
    {"region": "Cape Town, South Africa", "lat": -33.92, "lon": 18.42, "actual_drought_score": 3},
    {"region": "Sertão, Northeast Brazil", "lat": -7.2, "lon": -39.3, "actual_drought_score": 3},
    {"region": "Gobi Desert, Mongolia", "lat": 42.59, "lon": 103.43, "actual_drought_score": 3},
    {"region": "Andalusia, Spain", "lat": 37.38, "lon": -5.98, "actual_drought_score": 3},
    {"region": "Murray-Darling Basin, Australia", "lat": -33.0, "lon": 142.0, "actual_drought_score": 3},
    {"region": "Madagascar South", "lat": -24.2, "lon": 45.3, "actual_drought_score": 4},
    {"region": "Tigray, Ethiopia", "lat": 14.0, "lon": 38.5, "actual_drought_score": 4},
    {"region": "Chihuahua, Mexico", "lat": 28.6, "lon": -106.0, "actual_drought_score": 3},

    # Moderate/Mild Stress Zones (Score 1-2)
    {"region": "Maharashtra, India", "lat": 19.75, "lon": 75.71, "actual_drought_score": 2},
    {"region": "Central Texas, USA", "lat": 31.0, "lon": -100.0, "actual_drought_score": 2},
    {"region": "Po Valley, Italy", "lat": 45.0, "lon": 10.0, "actual_drought_score": 2},
    {"region": "Mato Grosso, Brazil", "lat": -13.0, "lon": -56.0, "actual_drought_score": 1},
    {"region": "Saskatchewan, Canada", "lat": 51.0, "lon": -106.0, "actual_drought_score": 1},
    {"region": "Punjab, India", "lat": 31.1, "lon": 75.3, "actual_drought_score": 1},
    {"region": "Rhine Valley, Germany", "lat": 49.5, "lon": 8.0, "actual_drought_score": 1},
    {"region": "Anatolia, Turkey", "lat": 39.0, "lon": 33.0, "actual_drought_score": 2},
    {"region": "Buenos Aires Province, Argentina", "lat": -36.0, "lon": -60.0, "actual_drought_score": 2},
    {"region": "Great Plains, USA", "lat": 41.0, "lon": -100.0, "actual_drought_score": 2},

    # Healthy Agricultural & Urban Zones (Score 0)
    {"region": "London, UK", "lat": 51.5, "lon": -0.12, "actual_drought_score": 0},
    {"region": "Tokyo, Japan", "lat": 35.67, "lon": 139.65, "actual_drought_score": 0},
    {"region": "Paris, France", "lat": 48.85, "lon": 2.35, "actual_drought_score": 0},
    {"region": "New York, USA", "lat": 40.71, "lon": -74.0, "actual_drought_score": 0},
    {"region": "Seoul, South Korea", "lat": 37.56, "lon": 126.97, "actual_drought_score": 0},
    {"region": "Auckland, New Zealand", "lat": -36.84, "lon": 174.76, "actual_drought_score": 0},
    {"region": "Kerala, India", "lat": 10.85, "lon": 76.27, "actual_drought_score": 0},
    {"region": "Mekong Delta, Vietnam", "lat": 10.03, "lon": 105.78, "actual_drought_score": 0},
    {"region": "Java, Indonesia", "lat": -7.0, "lon": 110.0, "actual_drought_score": 0},
    {"region": "Guangdong, China", "lat": 23.1, "lon": 113.2, "actual_drought_score": 0},

    # Rainforests & Wetlands (Score 0)
    {"region": "Amazon Rainforest, Brazil", "lat": -3.0, "lon": -60.0, "actual_drought_score": 0},
    {"region": "Congo Basin, DRC", "lat": 0.0, "lon": 22.0, "actual_drought_score": 0},
    {"region": "Sundarbans, Bangladesh", "lat": 21.9, "lon": 89.0, "actual_drought_score": 0},
    {"region": "Daintree Rainforest, Australia", "lat": -16.1, "lon": 145.4, "actual_drought_score": 0},
    {"region": "Papua New Guinea Highlands", "lat": -6.0, "lon": 144.0, "actual_drought_score": 0},
    {"region": "Everglades, USA", "lat": 26.3, "lon": -80.9, "actual_drought_score": 0},
    {"region": "Borneo Rainforest, Malaysia", "lat": 2.0, "lon": 114.0, "actual_drought_score": 0},
    {"region": "Valdivian Forest, Chile", "lat": -40.0, "lon": -72.0, "actual_drought_score": 0},
    {"region": "Tongass Forest, Alaska", "lat": 57.0, "lon": -135.0, "actual_drought_score": 0},
    {"region": "Sinharaja Forest, Sri Lanka", "lat": 6.4, "lon": 80.4, "actual_drought_score": 0},

    # Tundra, Taiga, and Cold Zones (Score 0)
    {"region": "Siberian Tundra, Russia", "lat": 61.0, "lon": 100.0, "actual_drought_score": 0},
    {"region": "Yukon, Canada", "lat": 64.0, "lon": -135.0, "actual_drought_score": 0},
    {"region": "Nuuk, Greenland", "lat": 64.18, "lon": -51.72, "actual_drought_score": 0},
    {"region": "Svalbard, Norway", "lat": 78.2, "lon": 15.6, "actual_drought_score": 0},
    {"region": "Kamchatka, Russia", "lat": 56.0, "lon": 159.0, "actual_drought_score": 0},
    {"region": "Iceland Highlands", "lat": 64.9, "lon": -18.0, "actual_drought_score": 0},
    {"region": "Lapland, Finland", "lat": 68.0, "lon": 26.0, "actual_drought_score": 0},
    {"region": "Baffin Island, Canada", "lat": 69.0, "lon": -72.0, "actual_drought_score": 0},
    {"region": "Tierra del Fuego, Argentina", "lat": -54.0, "lon": -67.0, "actual_drought_score": 0},
    {"region": "Hokkaido, Japan", "lat": 43.0, "lon": 142.0, "actual_drought_score": 0},
    
    # ... Expanding with 40 more mixed global locations to hit 100
    {"region": "Hawaii, USA", "lat": 19.89, "lon": -155.58, "actual_drought_score": 0},
    {"region": "Fiji, Pacific Ocean", "lat": -17.7, "lon": 178.0, "actual_drought_score": 0},
    {"region": "Oahu, USA", "lat": 21.4, "lon": -158.0, "actual_drought_score": 0},
    {"region": "Mauritius", "lat": -20.3, "lon": 57.5, "actual_drought_score": 0},
    {"region": "Seychelles", "lat": -4.6, "lon": 55.4, "actual_drought_score": 0},
    {"region": "Maldives", "lat": 3.2, "lon": 73.2, "actual_drought_score": 0},
    {"region": "Tenerife, Spain", "lat": 28.2, "lon": -16.6, "actual_drought_score": 1},
    {"region": "Azores, Portugal", "lat": 37.7, "lon": -25.6, "actual_drought_score": 0},
    {"region": "Bermuda", "lat": 32.3, "lon": -64.7, "actual_drought_score": 0},
    {"region": "Cyprus", "lat": 35.1, "lon": 33.4, "actual_drought_score": 2},
    {"region": "Crete, Greece", "lat": 35.2, "lon": 24.9, "actual_drought_score": 2},
    {"region": "Sicily, Italy", "lat": 37.5, "lon": 14.0, "actual_drought_score": 2},
    {"region": "Corsica, France", "lat": 42.0, "lon": 9.0, "actual_drought_score": 1},
    {"region": "Okinawa, Japan", "lat": 26.5, "lon": 127.9, "actual_drought_score": 0},
    {"region": "Taiwan", "lat": 23.6, "lon": 120.9, "actual_drought_score": 0},
    {"region": "Luzon, Philippines", "lat": 16.0, "lon": 121.0, "actual_drought_score": 0},
    {"region": "Mindanao, Philippines", "lat": 8.0, "lon": 125.0, "actual_drought_score": 0},
    {"region": "Sulawesi, Indonesia", "lat": -2.0, "lon": 120.0, "actual_drought_score": 0},
    {"region": "Timor-Leste", "lat": -8.8, "lon": 125.7, "actual_drought_score": 1},
    {"region": "Tasmania, Australia", "lat": -41.4, "lon": 146.3, "actual_drought_score": 0},
    {"region": "South Island, NZ", "lat": -43.0, "lon": 170.0, "actual_drought_score": 0},
    {"region": "Patagonia, Argentina", "lat": -45.0, "lon": -68.0, "actual_drought_score": 2},
    {"region": "Falkland Islands", "lat": -51.7, "lon": -59.0, "actual_drought_score": 0},
    {"region": "South Georgia", "lat": -54.2, "lon": -36.5, "actual_drought_score": 0},
    {"region": "Antarctic Peninsula", "lat": -65.0, "lon": -60.0, "actual_drought_score": 0},
    {"region": "McMurdo Station, Antarctica", "lat": -77.8, "lon": 166.6, "actual_drought_score": 0},
    {"region": "Novaya Zemlya, Russia", "lat": 74.0, "lon": 56.0, "actual_drought_score": 0},
    {"region": "Bering Strait", "lat": 65.9, "lon": -168.9, "actual_drought_score": 0},
    {"region": "Aleutian Islands, USA", "lat": 52.0, "lon": -174.0, "actual_drought_score": 0},
    {"region": "Galapagos, Ecuador", "lat": -0.8, "lon": -91.1, "actual_drought_score": 1},
    {"region": "Easter Island, Chile", "lat": -27.1, "lon": -109.3, "actual_drought_score": 0},
    {"region": "Tahiti, French Polynesia", "lat": -17.6, "lon": -149.4, "actual_drought_score": 0},
    {"region": "Bora Bora", "lat": -16.5, "lon": -151.7, "actual_drought_score": 0},
    {"region": "Cook Islands", "lat": -21.2, "lon": -159.7, "actual_drought_score": 0},
    {"region": "Samoa", "lat": -13.7, "lon": -172.1, "actual_drought_score": 0},
    {"region": "Tonga", "lat": -21.1, "lon": -175.2, "actual_drought_score": 0},
    {"region": "Vanuatu", "lat": -15.3, "lon": 166.9, "actual_drought_score": 0},
    {"region": "Solomon Islands", "lat": -9.6, "lon": 160.1, "actual_drought_score": 0},
    {"region": "New Caledonia", "lat": -20.9, "lon": 165.6, "actual_drought_score": 0},
    {"region": "Guam, USA", "lat": 13.4, "lon": 144.7, "actual_drought_score": 0},
    # --- SURGE OF SCORE 3 (SEVERE DROUGHT / HIGH CRISIS) ---
    {"region": "Sonora, Mexico", "lat": 29.0, "lon": -110.9, "actual_drought_score": 3},
    {"region": "Nevada, USA", "lat": 38.8, "lon": -116.4, "actual_drought_score": 3},
    {"region": "Arizona, USA", "lat": 34.0, "lon": -111.9, "actual_drought_score": 3},
    {"region": "Central Chile (Valparaiso)", "lat": -33.0, "lon": -71.6, "actual_drought_score": 3},
    {"region": "Mendoza, Argentina", "lat": -32.8, "lon": -68.8, "actual_drought_score": 3},
    {"region": "Syrian Steppe", "lat": 34.8, "lon": 38.2, "actual_drought_score": 3},
    {"region": "Tigris-Euphrates Basin, Iraq", "lat": 33.3, "lon": 43.7, "actual_drought_score": 3},
    {"region": "Khorasan, Iran", "lat": 35.0, "lon": 59.0, "actual_drought_score": 3},
    {"region": "Balochistan, Pakistan", "lat": 28.4, "lon": 65.5, "actual_drought_score": 3},
    {"region": "Turkana, Kenya", "lat": 3.1, "lon": 35.5, "actual_drought_score": 3},
    {"region": "Afar Triangle, Ethiopia", "lat": 11.4, "lon": 41.5, "actual_drought_score": 3},
    {"region": "Gansu Province, China", "lat": 38.0, "lon": 102.0, "actual_drought_score": 3},
    {"region": "New South Wales (Inland), Australia", "lat": -32.0, "lon": 145.0, "actual_drought_score": 3},
    {"region": "Queensland (Outback), Australia", "lat": -23.0, "lon": 143.0, "actual_drought_score": 3},
    {"region": "Ceará, Brazil", "lat": -5.2, "lon": -39.2, "actual_drought_score": 3},

    # --- SURGE OF SCORE 2 (MODERATE DROUGHT / AG-STRESS) ---
    {"region": "Kansas (High Plains), USA", "lat": 38.5, "lon": -100.0, "actual_drought_score": 2},
    {"region": "Nebraska, USA", "lat": 41.5, "lon": -99.7, "actual_drought_score": 2},
    {"region": "Alberta Prairies, Canada", "lat": 51.0, "lon": -112.0, "actual_drought_score": 2},
    {"region": "Mato Grosso do Sul, Brazil", "lat": -20.4, "lon": -54.6, "actual_drought_score": 2},
    {"region": "Cordoba, Argentina", "lat": -31.4, "lon": -64.1, "actual_drought_score": 2},
    {"region": "Alentejo, Portugal", "lat": 38.0, "lon": -7.8, "actual_drought_score": 2},
    {"region": "Catalonia, Spain", "lat": 41.8, "lon": 1.5, "actual_drought_score": 2},
    {"region": "Thessaly, Greece", "lat": 39.5, "lon": 22.0, "actual_drought_score": 2},
    {"region": "Kazakh Steppe, Kazakhstan", "lat": 48.0, "lon": 68.0, "actual_drought_score": 2},
    {"region": "Kherson Oblast, Ukraine", "lat": 46.6, "lon": 32.6, "actual_drought_score": 2},
    {"region": "Gujarat, India", "lat": 22.2, "lon": 71.1, "actual_drought_score": 2},
    {"region": "North Karnataka, India", "lat": 15.3, "lon": 75.1, "actual_drought_score": 2},
    {"region": "Isan Region, Thailand", "lat": 16.0, "lon": 103.0, "actual_drought_score": 2},
    {"region": "Zambezi River Basin, Zambia", "lat": -15.0, "lon": 25.0, "actual_drought_score": 2},
    {"region": "Matabeleland, Zimbabwe", "lat": -19.0, "lon": 28.0, "actual_drought_score": 2},
    {"region": "Eastern Cape, South Africa", "lat": -32.0, "lon": 26.0, "actual_drought_score": 2}
]

print(f"Initiating Massive Planetary Ingestion: {len(global_training_zones)} Target Zones...")
print("Estimated execution time: ~4 minutes. Do not interrupt kernel.")
training_data = []

# --- 3. THE EXECUTION LOOP ---
for i, zone in enumerate(global_training_zones, 1):
    print(f"[{i}/{len(global_training_zones)}] Fetching telemetry for {zone['region']}...")
    
    try:
        features = fetch_drought_features(zone['lat'], zone['lon'])
        features['Target_Drought_Score'] = zone['actual_drought_score']
        features['Region_Name'] = zone['region'] 
        
        training_data.append(features)
        
        # The critical 2-second sleep to prevent API bans
        time.sleep(2) 
        
    except Exception as e:
        print(f"   [!] Skipping {zone['region']} due to API error: {e}")

# --- 4. COMPILE AND SAVE ---
global_df = pd.DataFrame(training_data)

if not global_df.empty:
    cols = [c for c in global_df.columns if c not in ['Region_Name', 'Target_Drought_Score']]
    global_df = global_df[['Region_Name'] + cols + ['Target_Drought_Score']]

    print(f"\n✅ 100-Point Planetary Dataset Assembled! ({len(global_df)} successful pulls)")
    display(global_df.head(10)) # Just showing the top 10 so it doesn't flood your screen

    global_df.to_csv("global_drought_training_data_v3_massive.csv", index=False)
    print("Saved to 'global_drought_training_data_v3_massive.csv'")
else:
    print("\n❌ CRITICAL: No data was collected. Please check your API connection.")

Initiating Massive Planetary Ingestion: 131 Target Zones...
Estimated execution time: ~4 minutes. Do not interrupt kernel.
[1/131] Fetching telemetry for Sahara Desert, Chad...
[2/131] Fetching telemetry for Atacama Desert, Chile...
[3/131] Fetching telemetry for Rub' al Khali, Saudi Arabia...
[4/131] Fetching telemetry for Namib Desert, Namibia...
[5/131] Fetching telemetry for Kalahari Desert, Botswana...
[6/131] Fetching telemetry for Simpson Desert, Australia...
[7/131] Fetching telemetry for Thar Desert, India...
[8/131] Fetching telemetry for Taklamakan Desert, China...
[9/131] Fetching telemetry for Mojave Desert, USA...
[10/131] Fetching telemetry for Karakum Desert, Turkmenistan...
[11/131] Fetching telemetry for Horn of Africa, Somalia...
[12/131] Fetching telemetry for California Central Valley, USA...
[13/131] Fetching telemetry for Cape Town, South Africa...
[14/131] Fetching telemetry for Sertão, Northeast Brazil...
[15/131] Fetching telemetry for Gobi Desert, Mongolia...

,Region_Name,PRECTOTCORR,PS,QV2M,T2M,T2MDEW,T2MWET,T2M_MAX,T2M_MIN,T2M_RANGE,TS,WS10M,Target_Drought_Score
0,"Sahara Desert, Chad",0.000000,98.181429,2.381429,21.825714,-6.907143,7.460000,28.541429,15.732857,12.808571,22.344286,8.267143,4
1,"Atacama Desert, Chile",0.635714,79.908571,10.167143,20.067143,10.747143,15.404286,26.997143,15.035714,11.961429,23.050000,3.278571,4
2,"Rub' al Khali, Saudi Arabia",0.000000,98.190000,3.234286,23.831429,-3.094286,10.367143,31.250000,17.205714,14.044286,24.412857,2.868571,4
3,"Namib Desert, Namibia",0.022857,96.920000,9.938571,20.554286,13.382857,16.968571,27.775714,15.261429,12.514286,23.342857,3.327143,4
4,"Kalahari Desert, Botswana",3.832857,88.494286,11.810000,26.384286,14.284286,20.335714,32.455714,20.607143,11.848571,27.655714,3.814286,4
5,"Simpson Desert, Australia",14.528571,98.882857,18.440000,28.662857,23.554286,26.111429,32.774286,26.020000,6.754286,29.402857,6.730000,4
6,"Thar Desert, India",0.000000,99.260000,5.241429,24.880000,3.845714,14.362857,32.128571,17.537143,14.591429,25.098571,2.481429,4
7,"Taklamakan Desert, China",0.000000,88.681429,1.438571,9.302857,-14.598571,-2.650000,18.265714,2.090000,16.175714,8.314286,2.784286,4
8,"Mojave Desert, USA",0.254286,91.662857,3.555714,11.630000,-3.278571,4.175714,18.575714,5.141429,13.434286,11.201429,2.690000,4
9,"Karakum Desert, Turkmenistan",0.277143,100.208571,3.431429,12.504286,-1.730000,5.388571,19.720000,6.181429,13.538571,12.527143,4.554286,4


Saved to 'global_drought_training_data_v3_massive.csv'


In [5]:
import pandas as pd
import xgboost as xgb
import numpy as np

# 1. Load and Train
print("Loading Expanded Global Dataset...")
df = pd.read_csv("global_drought_training_data_v2.csv")
X = df.drop(columns=['Region_Name', 'Target_Drought_Score'])
y = df['Target_Drought_Score']

model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=150,      
    learning_rate=0.05,     
    max_depth=5,           
    random_state=42
)

print("Training Advanced XGBoost Engine...")
model.fit(X, y)

# 2. THE BLIND TEST (Unseen Data)
print("\n🌍 Executing Blind Inference on Unseen Coordinates...")
test_region = "Alice Springs, Australian Outback"
test_lat, test_lon = -23.69, 133.88

try:
    # Fetch live weather for the Outback
    blind_features = fetch_drought_features(test_lat, test_lon)
    
    # Format exactly like our training features X
    blind_df = pd.DataFrame([blind_features])
    blind_df = blind_df[X.columns] 
    
    # Make the prediction
    blind_prediction = model.predict(blind_df)[0]
    
    print("-" * 50)
    print(f"TARGET REGION: {test_region}")
    print(f"LIVE TEMPERATURE (Avg): {blind_features['T2M']:.2f} °C")
    print(f"LIVE HUMIDITY: {blind_features['QV2M']:.2f} g/kg")
    print("-" * 50)
    print(f"🚨 AI PREDICTED DROUGHT RISK SCORE: {blind_prediction:.2f} / 4.0")
    print("-" * 50)
    
except Exception as e:
    print(f"Test failed due to API issue: {e}")

# Save the final production brain
model.save_model("production_global_drought_model.json")

Loading Expanded Global Dataset...
Training Advanced XGBoost Engine...

🌍 Executing Blind Inference on Unseen Coordinates...
--------------------------------------------------
TARGET REGION: Alice Springs, Australian Outback
LIVE TEMPERATURE (Avg): 26.67 °C
LIVE HUMIDITY: 16.37 g/kg
--------------------------------------------------
🚨 AI PREDICTED DROUGHT RISK SCORE: 2.98 / 4.0
--------------------------------------------------


In [6]:
import xgboost as xgb
import pandas as pd
import datetime
import requests

# --- 1. RELOAD THE SAVED MODEL ---
print("Loading the production model...")
loaded_model = xgb.XGBRegressor()
loaded_model.load_model("production_global_drought_model.json")

# --- 2. NASA API INGESTION FUNCTION ---
# (Keeping this here so the cell is completely self-contained)
def fetch_live_weather(lat, lon):
    end_date = datetime.datetime.now() - datetime.timedelta(days=2) 
    start_date = end_date - datetime.timedelta(days=7)
    
    start_str = start_date.strftime('%Y%m%d')
    end_str = end_date.strftime('%Y%m%d')
    
    parameters = "PRECTOTCORR,PS,QV2M,T2M,T2MDEW,T2MWET,T2M_MAX,T2M_MIN,T2M_RANGE,TS,WS10M"
    url = (f"https://power.larc.nasa.gov/api/temporal/daily/point?"
           f"parameters={parameters}&community=AG&longitude={lon}&latitude={lat}"
           f"&start={start_str}&end={end_str}&format=JSON")
    
    response = requests.get(url)
    if response.status_code != 200:
        raise Exception(f"NASA API Failed: {response.status_code}")
        
    data = response.json()['properties']['parameter']
    features = {}
    for param in parameters.split(','):
        daily_values = list(data[param].values())
        valid_values = [v for v in daily_values if v != -999.0]
        features[param] = sum(valid_values) / len(valid_values) if valid_values else 0.0
    return features

# --- 3. EXECUTE THE LIVE TEST ---
# Let's test Navi Mumbai, but feel free to change these!
test_region = "Navi Mumbai, India"
test_lat = 19.03
test_lon = 73.02

print(f"\n🌍 Pinging NASA Satellites for {test_region} (Lat: {test_lat}, Lon: {test_lon})...")

try:
    live_features = fetch_live_weather(test_lat, test_lon)
    
    # Format the data exactly how XGBoost expects it
    expected_columns = ["PRECTOTCORR","PS","QV2M","T2M","T2MDEW","T2MWET","T2M_MAX","T2M_MIN","T2M_RANGE","TS","WS10M"]
    df_input = pd.DataFrame([live_features])[expected_columns]
    
    # Run the Inference
    prediction = loaded_model.predict(df_input)[0]
    
    print("\n" + "="*50)
    print("📡 LIVE TELEMETRY RECEIVED:")
    print(f"   Avg Temperature : {live_features['T2M']:.2f} °C")
    print(f"   Max Temperature : {live_features['T2M_MAX']:.2f} °C")
    print(f"   Recent Rainfall : {live_features['PRECTOTCORR']:.2f} mm/day")
    print(f"   Specific Humidity : {live_features['QV2M']:.2f} g/kg")
    print("="*50)
    print(f"🚨 AI DROUGHT RISK SCORE: {prediction:.2f} / 4.0")
    print("="*50)

except Exception as e:
    print(f"Test failed: {e}")

Loading the production model...

🌍 Pinging NASA Satellites for Navi Mumbai, India (Lat: 19.03, Lon: 73.02)...

📡 LIVE TELEMETRY RECEIVED:
   Avg Temperature : 27.68 °C
   Max Temperature : 35.14 °C
   Recent Rainfall : 0.04 mm/day
   Specific Humidity : 10.63 g/kg
🚨 AI DROUGHT RISK SCORE: 1.96 / 4.0
